# Interactive Script: **Initial Data Cube**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
This notebook walks through building an EO data cube with stac2cube, and through the tools that read, edit, export and summarise it.

Everything here is also available without code in `interactive/User_Interface_Tools.ipynb` (Data Cube Builder + Data Cube Editor). Use this notebook when you want to script the workflow or understand what the interface does under the hood.

## Contents

1. [Before You Build: What Is Available](#1-before-you-build-what-is-available)
2. [Build an Initial Data Cube](#2-build-an-initial-data-cube)
3. [Which Scenes Go In](#3-which-scenes-go-in)
4. [Visualization Tools](#4-visualization-tools)
5. [Export the Cube](#5-export-the-cube)
6. [Read a Cube Back](#6-read-a-cube-back)
7. [Update a Data Cube](#7-update-a-data-cube)
8. [Clip and Reproject](#8-clip-and-reproject)
9. [Temporal Composites](#9-temporal-composites)
10. [Slice Time and Band](#10-slice-time-and-band)
11. [Statistics and Time Series Tools](#11-statistics-and-time-series-tools)
12. [Mosaic Several Cubes](#12-mosaic-several-cubes)
13. [Other Missions](#13-other-missions)

**Related notebooks:** cloud and shadow masking in `2_Cloudmask_Data_Cube.ipynb`, co-registration in `3_Coregister_Data_Cube.ipynb`, super-resolution in `4_Superresolve_Data_Cube.ipynb`, catalogue comparison in `data_availability.ipynb`.

---

In [ ]:
from stac2cube import (
    get_stac_layers,
    missions,
    show_parameter_help,
    open_cube,
    export_stac,
)
import xarray as xr
import numpy as np
import dask
from dask.diagnostics import ProgressBar

## 1. Before You Build: What Is Available

Two cheap metadata checks before you spend time on pixels. Both only read the STAC catalogue, no imagery is downloaded.

### 1.1 Scene footprints over your area

Sentinel-2 is delivered in fixed tiles, and an area can sit on the edge of a tile or be split between neighbouring orbits. The preview maps the outlines of the scenes covering the area, so you can see whether a single orbit already covers it, or whether the cube will be stitched from several tiles.

Returns a map, a per-orbit coverage table, and an info dict with short findings.

In [ ]:
from stac2cube import preview_scene_footprints

m, df, info = preview_scene_footprints(
    mission="sentinel_2_l2a",
    polygon="../polygons/test.gpkg",
    daterange=["2024-04-01", "2024-04-30"],
    coverage_geometry="bbox",   # "bbox" = the search rectangle, "polygon" = the drawn shape
)
df

In [ ]:
for note in info["notes"]:
    print("-", note)

In [ ]:
m   # the footprint map

### 1.2 Which projection your scenes come in

Sentinel-2 tiles are delivered in UTM zones. If the area straddles two zones, one of them has to be picked as the cube's grid. This lists the candidates with the share of the area each one natively covers, which is the same ranking the builder uses when you leave `crs=None`.

In [ ]:
from stac2cube import probe_native_crs

probe_native_crs("sentinel_2_l2a", "../polygons/test.gpkg")

> Scene availability differs between catalogues (element84, Planetary Computer, terrabyte, CDSE). To compare them for your area, see `data_availability.ipynb`.

## 2. Build an Initial Data Cube

### 2.1 Set the parameters

**Run the next cell for the full parameter documentation.**

In [ ]:
show_parameter_help(open_by_default=False)

In [ ]:
# Missions, their bands, indices and defaults.
missions()

In [ ]:
mission = "s2"                              # see missions() -> name or alias
polygon = "../polygons/test.gpkg"           # vector file, or WGS84 bbox [xmin, ymin, xmax, ymax]
resolution = 10                             # metres
daterange = ["2024-04-01", "2024-04-10"]
bands = ["blue", "green", "red", "nir"]
indices = ["ndvi", "ndwi"]
clip_raster = False                         # False = bounding box, True = cut to the polygon
max_cc = 100                                # STAC tile cloud cover; keep 100, filter later per scene
cloud_masking = False                       # SCL masking (see notebook 2 for s2cloudless)
source = "e84"                              # e84 (element84), pc (Planetary Computer), tb (terrabyte), cdse
output = None                               # None returns a lazy cube without computing

> **daterange also takes seasonal forms**, which repeat the window in every year of the archive:
> - `["04-01", "10-31"]` -> vegetation season, all available years
> - `{"season": ["04-01", "10-31"], "years": [2023, 2024]}` -> only those years
> - `{"season": ["04-01", "10-31"], "years": "2018-2024"}` -> a year range

#### (Optional) Draw the area on a map instead

Skip this if you already have a polygon file. Draw a rectangle with the tools at the top left, then run the cell after it.

> `m.user_roi` is the **last** shape drawn, so the cell below takes one area even if you drew several. For all of them use `m.draw_features`, write them to a GeoJSON `FeatureCollection` and point `polygon` at that file - several features switch the build into batch processing, one cube per feature (see below). The Data Cube Builder interface does this for you.

In [ ]:
from stac2cube import satellite_map

m = satellite_map(center=(41.042, 29.017), zoom=13, draw=True)
m

In [ ]:
# Takes the drawn shape as the new polygon (overwrites the polygon set above).
if m is not None and m.user_roi is not None:
    ring = m.user_roi["geometry"]["coordinates"][0]
    xs = [x for x, y in ring]
    ys = [y for x, y in ring]
    polygon = [min(xs), min(ys), max(xs), max(ys)]   # [xmin, ymin, xmax, ymax]
    print(polygon)
else:
    print("Nothing drawn on the map, keeping the polygon set above:", polygon)

#### (Optional) Build the area from a coordinate

`aoi_from_point` grows a rectangle of a given **pixel** size around a coordinate, the way `cubo` does: say where and how many pixels, and the cube comes out at exactly that size. `edge_size=128` is a square; `edge_size=(height, width)` is a rectangle.

Two things are worth knowing before you use it.

**The rectangle is snapped to the pixel grid**, not centred exactly on your coordinate. The centre then sits within half a pixel of the point you gave - at most 5 m at 10 m resolution, reported per point as `offset_m`. That half pixel buys two things exact centring cannot: the cube is read without resampling, because its grid coincides with the scenes', and squares from nearby points share one grid and can be compared pixel to pixel.

**It must be drawn in the same projection the cube will be built in.** Draw it in one UTM zone and build in another and the rectangle arrives rotated, its bounding box grows, and 128 px stops being 128 px. `probe_native_crs` (section 1.2) says which projection the scenes come in; pass that same value to `get_stac_layers(crs=...)`.

A bare point is a zero-area search box, so below the rectangle is drawn once to get a real extent, the catalogue is asked, and it is redrawn on that grid.

In [ ]:
from stac2cube import aoi_from_point, probe_native_crs

lat, lon = 41.0421, 29.0173          # decimal degrees, LATITUDE first

# Pass 1 only exists to give probe_native_crs something with an area to search.
first = aoi_from_point(lat, lon, edge_size=128, resolution=resolution, q=True)
target_crs = probe_native_crs(
    "sentinel_2_l2a",
    [float(v) for v in first.to_crs("EPSG:4326").total_bounds],
    source=source,
)[0]["crs"]
print("cubes over this area will be built in", target_crs)

point_aoi = aoi_from_point(
    lat, lon,
    edge_size=128,                   # pixels; (height, width) for a rectangle
    resolution=resolution,
    crs=target_crs,                  # the same value goes to get_stac_layers
    out="../polygons/point_aoi.gpkg",
)
point_aoi.drop(columns="geometry")

**Several points at once.** Pass sequences instead of scalars and you get one feature per point, which `get_stac_layers` then builds as a batch - one data cube per point. `read_point_file` loads them from a GIS file and converts whatever projection they are stored in to latitude/longitude, so a UTM or national-grid file needs no preparation. A file with no CRS at all is refused rather than assumed to be lon/lat.

`../polygons/test_points_multi.gpkg` holds four sample sites, spaced far enough apart that their cubes do not overlap.

In [ ]:
from stac2cube import read_point_file

lats, lons, file_crs = read_point_file("../polygons/test_points_multi.gpkg")
print(f"{len(lats)} points, file was stored in {file_crs}")

sites = aoi_from_point(
    lats, lons,
    edge_size=128,
    resolution=resolution,
    crs=target_crs,
    out="../polygons/sites_aoi.gpkg",
)
sites.drop(columns="geometry")

#### (Optional) Split a large area into tiles

`tile_aoi` cuts an area into a grid of equal pieces and writes them as one multi-feature file, so the build makes one cube per tile. `n_pieces=(2, 3)` fixes the grid; a plain `n_pieces=6` picks the factor pair whose tiles come out closest to square and tells you which it chose.

Tile edges land on whole pixels and on a shared grid, so neighbouring cubes line up exactly and can be mosaicked later without resampling (section 12).

Splitting is **not free**. The build is network-latency bound: cost tracks the number of dates and barely moves with area, so six tiles cost roughly six times the wall time of one cube over the same ground unless you run them in parallel. Split when you want independent jobs, a build you can restart, or files small enough to move around - not by default.

`estimate_piece_bytes` sizes the result before anything is downloaded. You have to tell it how many dates to assume, since that is only known after the search.

In [ ]:
from stac2cube import estimate_piece_bytes, tile_aoi

tiles = tile_aoi(
    polygon,
    n_pieces=(2, 3),                 # (rows, columns); or an int to auto-factor
    resolution=resolution,
    crs=target_crs,
    out="../polygons/test_tiled_2_3.gpkg",
)
est = estimate_piece_bytes(tiles, n_dates=100, n_bands=len(bands))
print(f"largest tile ~{est['largest'] / 1e9:.2f} GB, all of them ~{est['total'] / 1e9:.2f} GB"
      "  (assuming 100 dates, float32)")

tiles.drop(columns="geometry")

#### What a multi-feature area means for the build

Any polygon file with **more than one feature** switches `get_stac_layers` into batch processing - this is not specific to the two tools above, a hand-drawn file with several polygons does the same:

* it returns a **list** of cubes, one per feature, in file order;
* with `output="../results/test.nc"` it writes `test_01.nc`, `test_02.nc`, ... (zero-padded to at least two digits, so they sort the way you read them);
* each feature is built on its own exact grid, so a tile asked for as 128 x 128 px comes back as 128 x 128 px;
* `dates=` is not supported for a batch, because every feature has its own time axis.

Rectangles from `tile_aoi` and `aoi_from_point` **are** the area, so build them with `clip_raster=False`. Clipping a cube to its own edge does nothing.

The cell below is left commented out because it downloads six cubes.

In [ ]:
# stac_tiles = get_stac_layers(
#     mission=mission,
#     polygon="../polygons/test_tiled_2_3.gpkg",   # 6 features -> 6 data cubes
#     resolution=resolution,
#     daterange=daterange,
#     bands=bands,
#     max_cc=max_cc,
#     clip_raster=False,          # the tiles ARE the area
#     crs=target_crs,             # must match what tile_aoi drew in
#     source=source,
#     output="../results/tiled.nc",   # -> tiled_01.nc ... tiled_06.nc
# )
# len(stac_tiles), stac_tiles[0].sizes

### 2.2 Build

In [ ]:
stac = get_stac_layers(
    mission=mission,
    polygon=polygon,
    resolution=resolution,
    daterange=daterange,
    bands=bands,
    indices=indices,
    max_cc=max_cc,
    clip_raster=clip_raster,
    cloud_masking=cloud_masking,
    source=source,
    output=output,
)
stac

The cube's metadata lives in the attributes of the returned array: mission, bands, indices, CRS, transform, resolution, `cloud_status`, `stac_api` and the AOI bbox. Those attributes are what the update, masking and co-registration tools read later, so keep them with the cube.

In [ ]:
for k, v in stac.attrs.items():
    print(f"{k:26s} {v}")

**Rough size of the cube once computed:**

In [ ]:
def format_bytes(n):
    f = float(n)
    for u in ["B", "KiB", "MiB", "GiB", "TiB"]:
        if f < 1024 or u == "TiB":
            return f"{f:.2f} {u}"
        f /= 1024

print(format_bytes(stac.nbytes))

In [ ]:
# If the time coordinate is too long to print, use this and click
# "View as a scrollable element".
print(stac.time.values)

### 2.3 (Optional) Compute the cube

Lazy is the default: nothing is downloaded until a value is needed. Exporting computes it anyway, so this step is only worth it if you want the visualization tools below to respond instantly.

In [ ]:
if dask.is_dask_collection(stac):
    with ProgressBar():
        stac = stac.compute()
stac

### 2.4 Optional build parameters

These are set in the same `get_stac_layers` call. Full descriptions in `show_parameter_help()`.

| parameter | what it does |
|---|---|
| `crs` | Force a target projection, e.g. `"EPSG:32635"`. Default: the UTM zone natively covering most of the area. |
| `resampling_method` | How bands are put on the target grid: `nearest` (default), `bilinear`, `cubic`. SCL and QA layers always use nearest. |
| `scene_metadata` | Attach per-scene STAC properties as time coordinates. Sentinel-2 only. Available fields: `acq_datetime`, `sun_azimuth`, `sun_elevation`, `view_azimuth`, `incidence_angle`, `relative_orbit`, `processing_baseline`. Not every catalogue serves all of them: element84 and Planetary Computer carry no viewing angles. |
| `metadata_output` | Folder to download each scene's granule metadata XML into. Sentinel-2 only. |
| `compress` | zlib-compress the NetCDF. Lossless, slower write, much smaller for cloud-masked cubes. |
| `vrt` | Write a small `.vrt` next to the NetCDF so QGIS shows named bands and dates. |
| `export_settings` | Write the call as a config JSON next to the export. |
| `statistics_csv` | Write a per-band, per-date/month/year statistics table next to the export. |

In [ ]:
stac_meta = get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-10"],
    bands=["blue", "green", "red", "nir"],
    max_cc=100,
    crs="EPSG:32635",
    resampling_method="bilinear",
    scene_metadata=["sun_azimuth", "sun_elevation", "relative_orbit"],
    q=True,
)
print(stac_meta.sun_azimuth.values)
print(stac_meta.sun_elevation.values)
print(stac_meta.relative_orbit.values)

## 3. Which Scenes Go In

Four independent filters. They act at different stages, which is why they are separate parameters and not one setting.

### 3.1 Before the download: footprint prefilter

`min_footprint_coverage` drops acquisitions whose published outlines barely touch the area, before any pixel is fetched. It is a geometric estimate, so use it at a low threshold to skip scenes that only graze the area. It is the only filter that saves download time.

In [ ]:
stac_fp = get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-30"],
    bands=["red"],
    max_cc=100,
    min_footprint_coverage=0.05,   # skip scenes covering less than 5% of the area
    q=True,
)
print(stac_fp.time.values)

### 3.2 After the load: partial scenes

An acquisition can list the area as covered and still image only part of it, because the swath edge cuts across it. Those gaps load as missing pixels. `partial_scene_handling="remove"` drops scenes covering less than `min_scene_coverage` of the area, measured on the real pixels.

Every cube carries a `scene_coverage` time coordinate (fraction 0 to 1) whether or not you use the filter, so you can always inspect it first.

In [ ]:
print(stac.scene_coverage.values)

In [ ]:
stac_complete = get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-30"],
    bands=["red"],
    max_cc=100,
    partial_scene_handling="remove",
    min_scene_coverage=0.9,        # keep scenes imaging at least 90% of the area
    q=True,
)
print(stac_complete.time.values)

### 3.3 Cloud cover per scene

`max_cc` filters on the cloud percentage of the whole Sentinel-2 tile, which says little about your area. `scene_cloud_coverage` filters on the cloud percentage of your area, computed from the detected clouds, so it needs `cloud_masking=True` (or `keep_clouds=True`, which detects without masking).

In [ ]:
stac_clear = get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-30"],
    bands=["red"],
    max_cc=100,
    cloud_masking=True,
    scene_cloud_coverage=20,       # keep scenes with at most 20% cloud over the area
    q=True,
)
print(stac_clear.time.values)
print(stac_clear.cloud_percentage.values)

### 3.4 Named dates

`dates` keeps exactly the acquisitions you list, applied after the cube is built. Single cube only, and not available in update mode.

In [ ]:
stac_two = get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-10"],
    bands=["red"],
    max_cc=100,
    dates=["2024-04-01", "2024-04-06"],
    q=True,
)
print(stac_two.time.values)

### 3.5 Tiles that split the area

When the area sits across the north-south seam of two Sentinel-2 tiles, both tiles are acquired on the same day and are mosaicked into one scene by default. `tile_handling="separate"` keeps them as separate timesteps instead, with a `tile` coordinate. Sentinel-2 L2A only, and not combinable with update or composites.

In [ ]:
# stac_sep = get_stac_layers(..., tile_handling="separate")
# print(stac_sep.tile.values)

## 4. Visualization Tools

The viewers work on lazy cubes too. A lazy cube reloads on every date change, so it responds more slowly; compute it first (2.3) for an instant response.

In [ ]:
from stac2cube import interactive_time_view, save_timeseries_gif
import matplotlib.pyplot as plt

### 4.1 Interactive time view

Display modes: `rgb`, `false_color`, `ndvi`, `ndwi`. The index modes need the index in the cube.

`renderer="static"` draws with matplotlib; `renderer="interactive"` uses plotly, which gives pan, zoom and pixel readout.

In [ ]:
interactive_time_view(stac=stac, widget_type="slider")

In [ ]:
interactive_time_view(
    stac=stac,
    widget_type="dropdown",
    renderer="interactive",
    show_coords=True,
)

### 4.2 Animation

`display_mode="band"` renders one band in grey levels, `display_mode="custom"` maps any three bands to red, green and blue.

In [ ]:
save_timeseries_gif(
    da=stac,
    out_path="../animations/test_animation.gif",
    display_mode="rgb",
    fps=3,          # higher = faster animation
    label=True,     # date label burned into the frames
)

In [ ]:
# Free band mapping, QGIS style
save_timeseries_gif(
    da=stac,
    out_path="../animations/test_falsecolor.gif",
    display_mode="custom",
    rgb_bands=("nir", "red", "green"),
    fps=3,
)

## 5. Export the Cube

Three containers. The format follows the extension of `output`, or set `export_format` explicitly.

| format | when |
|---|---|
| NetCDF (`.nc`) | the default. One file, holds the full (time, band, y, x) cube. |
| Zarr (`.zarr`) | a chunked store (a folder). Written and read one scene at a time, so it stays cheap on memory for very large cubes. Compressed by default. |
| COGs (folder) | Cloud-Optimized GeoTIFFs, one file per date. GeoTIFF has no time dimension, so the time series is split. Band names survive in QGIS. |

> **Check your free disk space before exporting a long time series.**

In [ ]:
export_stac(stac, "../results/test.nc")

**Compression and the QGIS band labels**

`compress=True` applies lossless zlib to a NetCDF. Values read back bit-identical; cloud-masked cubes shrink a lot because the masked areas compress strongly.

`vrt=True` writes a small `.vrt` next to the NetCDF. GDAL flattens time and band into one numbered stack, so a 4-date, 6-band cube opens in QGIS as "Band 1 ... Band 24". The VRT relabels them as "2024-04-01 blue" and so on. It points at the cube, it does not copy data, so keep the `.nc` and the `.vrt` together.

In [ ]:
export_stac(stac, "../results/test_compressed.nc", compress=True, vrt=True)

**Zarr**

In [ ]:
export_stac(stac, "../results/test.zarr")

**Cloud-Optimized GeoTIFFs**

In [ ]:
from stac2cube import export_to_cogs

export_to_cogs(stac, "../results/cogs/")

**Exporting straight from the build**

Passing `output` to `get_stac_layers` builds and writes in one step, which is what you want on an HPC job. `export_settings=True` writes the call as a JSON config next to the cube, so the same cube can be rebuilt later, and `statistics_csv=True` writes a per-band statistics table.

In [ ]:
get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-10"],
    bands=["blue", "green", "red", "nir"],
    indices=["ndvi"],
    max_cc=100,
    output="../results/test_direct.nc",
    export_settings=True,
    statistics_csv=True,
    q=True,
)

## 6. Read a Cube Back

`open_cube` is the single reader for both formats: `.nc` opens with xarray, `.zarr` opens lazily as stored. Cubes written by older versions of stac2cube (variable `Spectral_Temporal_Stack`) are renamed to `Time_Series` on read, so old files keep working.

In [ ]:
from stac2cube import open_cube

with open_cube("../results/test.nc") as ds:
    stac = ds["Time_Series"].load()

stac

In [ ]:
# The same call works on a Zarr store.
with open_cube("../results/test.zarr") as ds:
    stac_zarr = ds["Time_Series"]
    print(stac_zarr.dims, stac_zarr.shape)

**Whole dataset instead of one variable**

A cube exported with temporal composites holds one variable per composite next to `Time_Series`.

In [ ]:
with open_cube("../results/test.nc") as ds:
    dataset = ds.load()

print(list(dataset.data_vars))

## 7. Update a Data Cube

Two reasons to update instead of rebuilding:

1. Extending the date range and re-running the build recomputes everything from scratch. Updating fetches only the missing dates.
2. On a machine that cannot hold a long time series, build a short cube and extend it step by step, one year at a time.

The stored cube's own parameters are restored, so `mission`, `resolution`, `polygon`, `bands`, `indices`, `crs` and `source` are taken from the file and cannot be changed here. Passing `bands` is the one exception: bands not yet in the cube are added and masked the same way as the existing ones.

The cube's **grid** is restored too, read straight off its own x and y coordinates rather than re-derived from the area. So the new dates land on exactly the pixels the cube already has, and a cube that no longer sits on its original bbox grid, because it was clipped, co-registered or reprojected, can still be extended. If the fresh query cannot be put on that grid, the update stops with an error rather than merging two grids into one oversized cube.

In [ ]:
img = "../results/test.nc"

with open_cube(img) as ds:
    print(ds["Time_Series"].shape, ds["Time_Series"].time.values)

In [ ]:
stac_updated = get_stac_layers(
    update=img,
    daterange=["2024-04-01", "2024-04-20"],   # the new, wider range
    cloud_masking=False,                      # must match how the cube was built
    output=None,                              # None returns the cube, a path overwrites the file
)
print(stac_updated.shape)                     # y and x unchanged, only time grew
stac_updated.time.values

> **Not available in update mode:** `dates`, `scene_cloud_coverage`, `partial_scene_handling="remove"`, `crs`, `scene_metadata`, `tile_handling="separate"`, `shadow_masking` and `keep_timeseries=False`. Each of them would either drop scenes already stored or change the grid the cube sits on. Build in one pass instead, or edit the finished cube.

**Adding a band to an existing cube**

List the bands you want, including the ones already there. The missing ones are computed for the dates already stored and appended. They arrive with the same cloud treatment as the stored bands, because the fresh query reuses the cube's own masking strategy.

In [ ]:
stac_more_bands = get_stac_layers(
    update=img,
    bands=["blue", "green", "red", "nir", "swir16"],   # swir16 is new
    output=None,
    q=True,
)
print(stac_more_bands.band.values)

> A cube storing exactly **one** index currently cannot be updated: the single-element list comes back from the file as a plain string and the update raises `AttributeError: 'str' object has no attribute 'tolist'`. Build with either no index or two or more until this is fixed.

## 8. Clip and Reproject

### 8.1 Clip

Both a lazy cube and an exported file can be clipped. The polygon can be a vector file in any projection, or a WGS84 bbox list `[xmin, ymin, xmax, ymax]` (a projected coordinate list is not accepted).

In [ ]:
from stac2cube import clip_stac

with open_cube("../results/test.nc") as ds:
    stac = ds["Time_Series"].load()

stac_clipped = clip_stac(stac=stac, polygon="../polygons/test_clip.gpkg")
print(stac.shape, "->", stac_clipped.shape)

In [ ]:
export_stac(stac_clipped, "../results/test_clipped.nc")

### 8.2 Reproject

Warps the cube into another CRS. The target must be a projected, metre-based CRS.

Reprojection resamples: both the values and the pixel grid change, and it is not reversible. Reproject once from the native cube rather than chaining warps. The result is the bounding box of the rotated footprint, so its corners are empty. `cloud_percentage` and `scene_coverage` are carried over unchanged, not recomputed, because the new corner gaps are neither cloud nor missing scene data.

In [ ]:
from stac2cube import reproject_stac

stac_36n = reproject_stac(stac, crs="EPSG:32636", resampling="nearest")
print(stac.attrs["crs"], "->", stac_36n.attrs["crs"])
print(stac_36n.attrs["reprojected_from"])

## 9. Temporal Composites

`stats` adds one variable per composite next to the time series. It can be set while building (chapter 2) or applied to a finished cube here. Composites are computed after the scene filters, so they always describe the scenes that survived.

Ready-made names combine an operation (`mean`, `median`, `min`, `max`, `std`) with a period:

- `*_timeseries` over the whole series
- `*_monthly` one per month present
- `*_annual` one per year present
- `*_all` all three at once

Add `keep_timeseries=False` in the build call to write the composites alone, without the time series. Such a cube has no time axis, so it cannot be updated, masked or co-registered afterwards.

In [ ]:
from stac2cube import calculate_statistics

stac_stats = calculate_statistics(stac, stats=["mean_all", "median_timeseries", "std_timeseries"])
print(list(stac_stats.data_vars))

In [ ]:
stac_stats.mean_timeseries

### 9.1 Custom composites

A composite can also be a dict, which lets you name your own window. `op` is the statistic, `name` becomes the variable name.

- **Season, repeating every year the cube covers:** `{"op": "mean", "season": ["04-01", "06-21"], "name": "spring_mean"}` gives `spring_mean_2024`, `spring_mean_2025`, ...
- **Restricted to given years:** add `"years": [2024, 2025]`
- **A single window, no year suffix:** `{"op": "median", "window": ["2024-04-01", "2024-04-06"], "name": "early_april"}`

Both ends include the whole day. A season that starts later than it ends (`["12-01", "02-28"]`) runs over New Year and is labelled by its start year. Each custom variable carries `composite_*` attributes recording the window, the number of contributing scenes and the first and last date, so a window only partly covered by the cube stays visible.

In [ ]:
stac_custom = calculate_statistics(
    stac,
    stats=[
        {"op": "median", "window": ["2024-04-01", "2024-04-06"], "name": "early_april"},
        {"op": "mean", "season": ["04-01", "06-21"], "name": "spring_mean"},
    ],
)
print(list(stac_custom.data_vars))
print(stac_custom["early_april"].attrs)

Export the whole dataset to keep the time series and every composite in one file.

`calculate_statistics` returns a Dataset whose georeferencing sits on the variables, not on the Dataset itself, and `export_stac` reads it from the object it is handed. Copy the two attributes across first, otherwise the export raises `'Dataset' object has no attribute 'crs'`. Composites requested through `stats=` in the build call are exported for you and need none of this.

In [ ]:
stac_stats.attrs["crs"] = stac.attrs["crs"]
stac_stats.attrs["transform"] = stac.attrs["transform"]

export_stac(stac_stats, "../results/test_stats.nc")

## 10. Slice Time and Band

Plain xarray. Once sliced, export with `export_stac` as in chapter 5.

### 10.1 Time

In [ ]:
# One date
stac_single_date = stac.sel(time="2024-04-01")

In [ ]:
# Several named dates (not a range)
stac_multi_dates = stac.sel(time=["2024-04-01", "2024-04-06"])
stac_multi_dates.time.values

In [ ]:
# A range
stac_time_range = stac.sel(time=slice("2024-04-01", "2024-04-06"))
stac_time_range.time.values

### 10.2 Bands

In [ ]:
stac_ndvi = stac.sel(band="ndvi")
stac_rgb = stac.sel(band=["red", "green", "blue"])
stac_rgb.band.values

### 10.3 Both

In [ ]:
stac_sub = stac.sel(time=slice("2024-04-01", "2024-04-06")).sel(band=["red", "green", "blue"])
stac_sub

## 11. Statistics and Time Series Tools

### 11.1 Statistics table of an exported cube

One CSV with a row per band and period. The `period` column says what the `label` names: `date` (every acquisition), `year`, or `month`. The year and month rows are computed from the scenes themselves, not summarised from the date rows, so they stay correct when the number of valid pixels varies between dates, which cloud masking guarantees it does.

Columns: `period, label, band, n_dates, mean, median, min, max, std`, plus `cloud_percentage` and `scene_coverage` where the cube carries them.

In [ ]:
from stac2cube import export_cube_statistics

df_stats = export_cube_statistics("../results/test.nc")
df_stats.head(12)

> `median` cannot be streamed, it needs a whole period in memory at once, so the peak is roughly the largest year of the cube. Drop it with `stats=["mean", "min", "max", "std"]` to keep the run streaming scene by scene.

### 11.2 Time series graph

Averaging over x and y computes the cube, so a lazy cube takes a while here. Read an exported file instead for a fast response.

In [ ]:
with open_cube("../results/test.nc") as ds:
    da = ds["Time_Series"].load()

da_sel = da.sel(band="ndvi").mean(dim=["x", "y"])

plt.figure(figsize=(12, 5))
plt.plot(da_sel["time"], da_sel, marker="o", label="Mean NDVI")
plt.xlabel("Time")
plt.ylabel("Mean NDVI")
plt.title("Mean NDVI over time")
plt.legend()
plt.grid()
plt.show()

**Smoothed series** (only meaningful on a long time series)

In [ ]:
smoothed = da_sel.rolling(time=7, center=True).mean()

plt.figure(figsize=(12, 5))
plt.plot(smoothed["time"], smoothed, label="Mean NDVI, 7-scene rolling mean")
plt.xlabel("Time")
plt.ylabel("Mean NDVI")
plt.legend()
plt.grid()
plt.show()

### 11.3 Seasonal-trend decomposition (LOESS)

Needs a long time series, several years at least, and the optional `statsmodels` package (`pip install statsmodels`), which is not a stac2cube dependency.

`statsmodels` expects a constant time step, so the irregular acquisition dates are resampled to daily and interpolated first. That interpolation invents values between acquisitions: it is a requirement of the method, not a property of the data, so read the components accordingly.

In [ ]:
import pandas as pd

try:
    from statsmodels.tsa.seasonal import MSTL
except ImportError:
    MSTL = None
    print("statsmodels is not installed, skipping. Install it with: pip install statsmodels")

ts = da.sel(band="ndvi").mean(dim=["x", "y"]).to_series()
ts.index = ts.index.floor("D")
ts_reg = ts.resample("D").interpolate(method="linear")

seasonal_periods = [30, 90, 365]        # monthly, quarterly, annual
if MSTL is None:
    pass
elif len(ts_reg) > 2 * max(seasonal_periods):
    result = MSTL(ts_reg, periods=seasonal_periods).fit()
    result.plot()
    plt.tight_layout()
    plt.show()
else:
    print(
        f"Time series too short for these periods "
        f"({len(ts_reg)} days, needs > {2 * max(seasonal_periods)}). "
        "Build a multi-year cube first."
    )

## 12. Mosaic Several Cubes

Merges cubes into one covering their combined extent. Useful when a large area was built in pieces, or when neighbouring areas should end up in one file. Cubes do not have to overlap; where none of them reaches, the mosaic is empty.

In [ ]:
from stac2cube import mosaic_layers, mosaic_cubes

First check what the cubes have in common. This reads metadata only, no pixels.

In [ ]:
cubes = ["../results/test.nc", "../results/test_clipped.nc"]

info = mosaic_layers(cubes)
print("common layers :", info["common"])
print("only in some  :", info["only_some"])
print("projections   :", info["crs_counts"])

In [ ]:
mosaic = mosaic_cubes(
    cubes,
    overlap="first",     # where both cover a pixel, take it from the first cube listed
    time_join="outer",   # keep every date any cube has
    band_join="inner",   # keep the bands all cubes share
    source_map=True,     # add a raster showing which cube each pixel came from
    output="../results/test_mosaic.nc",
)
mosaic

> `overlap="first"` is the only policy that never produces a value none of the inputs contained. `mean`, `median`, `min` and `max` reduce across the covering cubes instead. Cubes that are not already on the same grid are refused rather than silently resampled; pass `on_grid_mismatch="resample"` if you accept the resampling.

## 13. Other Missions

### 13.1 Sentinel-1 RTC

In [ ]:
missions().iloc[2]

In [ ]:
stac_s1 = get_stac_layers(
    mission="s1",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-30"],
    bands=["vh", "vv"],
    indices=["vh/vv", "rvi"],
    clip_raster=False,
    output=None,
)
stac_s1

In [ ]:
def normalize(band, clip_percentile=2):
    lo, hi = np.nanpercentile(band, [clip_percentile, 100 - clip_percentile])
    if hi == lo:
        return np.zeros_like(band)
    out = (np.clip(band, lo, hi) - lo) / (hi - lo)
    return np.nan_to_num(out, nan=0.5)

In [ ]:
vv = stac_s1.isel(time=0).sel(band="vv")

plt.figure(figsize=(10, 5))
plt.imshow(normalize(vv.values), cmap="gray")
plt.title(f"VV, {str(vv.time.values)[:10]}")
plt.axis("off")
plt.show()

**Select by orbit state**

`where()` drops the CRS information from the array, so write it back before exporting, otherwise the file has no projection in QGIS.

In [ ]:
stac_asc = stac_s1.where(stac_s1.orbit_state == "ascending", drop=True)
stac_asc = stac_asc.rio.write_crs(stac_s1.attrs["crs"])
stac_asc.time.values

### 13.2 Landsat Collection 2 Level-2

In [ ]:
missions().iloc[3]

In [ ]:
stac_ls = get_stac_layers(
    mission="l_oli",
    polygon="../polygons/test.gpkg",
    resolution=30,
    daterange=["2024-01-01", "2024-06-30"],
    bands=["blue", "green", "red", "nir"],
    indices=["ndvi"],
    max_cc=100,
    cloud_masking=False,
    output=None,
)
stac_ls.time.values

In [ ]:
rgb = stac_ls.isel(time=0).sel(band=["red", "green", "blue"]).values
composite = np.dstack([normalize(b) for b in rgb])

plt.figure(figsize=(10, 5))
plt.imshow(composite)
plt.title("Landsat, normalized RGB")
plt.axis("off")
plt.show()

### 13.3 COP DEM GLO-30

> **This mission does not currently work.** The elevation model and its topographic derivatives (slope, aspect, flow accumulation, TWI) are listed in `missions()`, but the WhiteboxTools backend behind them is switched off in this version and the build path still refers to it. Calling `get_stac_layers(mission="cop_dem", ...)` raises `NameError: name 'dem' is not defined`.
>
> The cell below is left commented out on purpose. Use another DEM source until this is re-enabled.

In [ ]:
missions().iloc[4]

In [ ]:
# Not functional in this version, see the note above.
#
# dem = get_stac_layers(
#     mission="cop_dem",
#     polygon="../polygons/test.gpkg",
#     clip_raster=False,
#     output=None,
#     topographic_features=["slope", "aspect"],
# )